### RRF Rerank

RRF（Reciprocal Rank Fusion） 是一种 无模型、无训练的排序融合算法。

> 专业项目中常将 **RRF 作为召回融合层** → 再用 **Cross-Encoder / LLM Reranker** 做精排。

**典型流程示例**

```mermaid
flowchart LR
    A[Query] --> B[BM25]
    A --> C[Embeddings]
    B --> D[RRF Rank]
    C --> D[RRF Rank]
    D --> E[Cross-Encoder / LLM Reranker]
```

In [7]:
from collections import defaultdict

def rrf_rerank(documnents: list[list[str]], k: int = 60, limit: int | None = None) -> list[tuple[str, float]]:
    """Reciprocal Rank Fusion (RRF)"""
    scores = defaultdict(float)
    for ranked in documnents:
        for rank, doc_id in enumerate(ranked, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:limit] if limit else ranked

**使用示例**

参数说明：
- `documnents`: 多个有序结果列表
- `k`: 平滑参数（通常 50~100）
- `limit`: 返回 top-N（None 表示全量）

假设我们已经用 BM25、BGE、Qwen 得到了召回结果

In [8]:
# BM25 排名结果
bm25_ranked = ["docA", "docB", "docC", "docD"]
# BGE Embedding 排名结果
bge_embed_ranked = ["docB", "docE", "docA", "docF"]
# Qwen Embedding 排名结果
qwen_embed_ranked = ["docC", "docA", "docG", "docH"]

merged_ranked_data = [bm25_ranked, bge_embed_ranked, qwen_embed_ranked]

In [9]:
# 使用默认的 k=60
rrf_rank_result = rrf_rerank(merged_ranked_data)
print("RRF result (default k=60):")
for doc_id, score in rrf_rank_result:
    print(f"  Doc ID: {doc_id}, Score: {score:.4f}")


RRF result (default k=60):
  Doc ID: docA, Score: 0.0484
  Doc ID: docB, Score: 0.0325
  Doc ID: docC, Score: 0.0323
  Doc ID: docE, Score: 0.0161
  Doc ID: docG, Score: 0.0159
  Doc ID: docD, Score: 0.0156
  Doc ID: docF, Score: 0.0156
  Doc ID: docH, Score: 0.0156


In [10]:
# 指定 k 值和 limit
rrf_rank_result = rrf_rerank(merged_ranked_data, k=20, limit=3)
print("RRF result (k=20, limit=3):")
for doc_id, score in rrf_rank_result:
    print(f"  Doc ID: {doc_id}, Score: {score:.4f}")

RRF result (k=20, limit=3):
  Doc ID: docA, Score: 0.1366
  Doc ID: docB, Score: 0.0931
  Doc ID: docC, Score: 0.0911
